# Netflix Data Analysis

## 1. Introduction
This project explores the Netflix Movies & TV Shows dataset to understand content trends,
genre distribution, ratings, and growth over time using Python (Pandas, Matplotlib, Seaborn).

**Objective:** Perform data cleaning, exploratory data analysis (EDA), and visualization
to extract meaningful insights about Netflix's content library.


## 2. Dataset
Source: [Netflix Movies and TV Shows - Kaggle](https://www.kaggle.com/datasets/shivamb/netflix-shows)

Download `netflix_titles.csv` from Kaggle and place it in this project folder as `data.csv`.

Columns include: show_id, type, title, director, cast, country, date_added, release_year,
rating, duration, listed_in (genres), description.


## 3. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

df = pd.read_csv("data.csv")
df.head()


## 4. Data Cleaning

In [ ]:
# Basic info
print(df.shape)
df.info()


In [ ]:
# Remove duplicate rows
df = df.drop_duplicates()

# Handle missing values
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
df.dropna(subset=['date_added', 'rating', 'duration'], inplace=True)

# Convert date columns
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month_name()

# Fix data types
df['release_year'] = df['release_year'].astype(int)

# Split duration into numeric value + unit (movies = minutes, TV shows = seasons)
df['duration_int'] = df['duration'].str.extract('(\d+)').astype(float)
df['duration_type'] = df['duration'].str.extract('([a-zA-Z]+)')

print("Missing values after cleaning:")
print(df.isnull().sum())


## 5. Data Understanding

In [ ]:
df.describe(include='all').T


## 6. Exploratory Data Analysis
Questions we'll answer:
- Which country produces the most content?
- Movies vs TV Shows split?
- Growth of content over the years?
- Most common genres?
- Longest movies?
- Rating distribution?
- Duration distribution?
- Top directors?


## 7. Visualizations

In [ ]:
# 1. Movies vs TV Shows (Count Plot)
plt.figure()
sns.countplot(data=df, x='type', palette='Set2')
plt.title('Movies vs TV Shows')
plt.savefig('images/01_movies_vs_tvshows.png', bbox_inches='tight')
plt.show()


In [ ]:
# 2. Content added per year (Line Plot)
yearly = df['year_added'].value_counts().sort_index()
plt.figure()
plt.plot(yearly.index, yearly.values, marker='o')
plt.title('Growth of Netflix Content Over the Years')
plt.xlabel('Year Added')
plt.ylabel('Number of Titles')
plt.savefig('images/02_growth_over_years.png', bbox_inches='tight')
plt.show()


In [ ]:
# 3. Top 15 countries (Bar Plot)
top_countries = df['country'].value_counts().head(15)
plt.figure()
sns.barplot(x=top_countries.values, y=top_countries.index, palette='viridis')
plt.title('Top 15 Countries by Content Count')
plt.xlabel('Number of Titles')
plt.savefig('images/03_top_countries.png', bbox_inches='tight')
plt.show()


In [ ]:
# 4. Rating distribution (Count Plot)
plt.figure()
order = df['rating'].value_counts().index
sns.countplot(data=df, y='rating', order=order, palette='coolwarm')
plt.title('Distribution of Content Ratings')
plt.savefig('images/04_rating_distribution.png', bbox_inches='tight')
plt.show()


In [ ]:
# 5. Duration distribution for Movies (Histogram)
movies = df[df['type'] == 'Movie']
plt.figure()
sns.histplot(movies['duration_int'], bins=30, kde=False, color='steelblue')
plt.title('Distribution of Movie Durations (minutes)')
plt.xlabel('Duration (min)')
plt.savefig('images/05_movie_duration_hist.png', bbox_inches='tight')
plt.show()


In [ ]:
# 6. Movie duration KDE Plot
plt.figure()
sns.kdeplot(movies['duration_int'], fill=True, color='purple')
plt.title('Movie Duration Density')
plt.xlabel('Duration (min)')
plt.savefig('images/06_movie_duration_kde.png', bbox_inches='tight')
plt.show()


In [ ]:
# 7. Box Plot - Movie duration outliers
plt.figure()
sns.boxplot(x=movies['duration_int'], color='orange')
plt.title('Movie Duration - Outlier Detection')
plt.savefig('images/07_movie_duration_box.png', bbox_inches='tight')
plt.show()


In [ ]:
# 8. Top 10 Genres (Bar Plot)
genres = df['listed_in'].str.split(', ').explode()
top_genres = genres.value_counts().head(10)
plt.figure()
sns.barplot(x=top_genres.values, y=top_genres.index, palette='mako')
plt.title('Top 10 Genres on Netflix')
plt.xlabel('Number of Titles')
plt.savefig('images/08_top_genres.png', bbox_inches='tight')
plt.show()


In [ ]:
# 9. Top 10 Directors (Bar Plot, excluding Unknown)
top_directors = df[df['director'] != 'Unknown']['director'].value_counts().head(10)
plt.figure()
sns.barplot(x=top_directors.values, y=top_directors.index, palette='crest')
plt.title('Top 10 Directors by Number of Titles')
plt.xlabel('Number of Titles')
plt.savefig('images/09_top_directors.png', bbox_inches='tight')
plt.show()


In [ ]:
# 10. Release year trend (Line Plot)
release_trend = df['release_year'].value_counts().sort_index()
release_trend = release_trend[release_trend.index >= 1990]
plt.figure()
plt.plot(release_trend.index, release_trend.values, color='darkred')
plt.title('Titles Released per Year (1990+)')
plt.xlabel('Release Year')
plt.ylabel('Count')
plt.savefig('images/10_release_year_trend.png', bbox_inches='tight')
plt.show()


In [ ]:
# 11. Movies vs TV Shows over years (Line Plot)
type_year = df.groupby(['year_added', 'type']).size().unstack(fill_value=0)
plt.figure()
type_year.plot(marker='o')
plt.title('Movies vs TV Shows Added Over Years')
plt.xlabel('Year Added')
plt.ylabel('Count')
plt.savefig('images/11_type_over_years.png', bbox_inches='tight')
plt.show()


In [ ]:
# 12. Correlation Heatmap (numeric columns)
plt.figure()
numeric_df = df[['release_year', 'duration_int']].dropna()
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.savefig('images/12_correlation_heatmap.png', bbox_inches='tight')
plt.show()


In [ ]:
# 13. Top 10 longest movies (Bar Plot)
longest = movies.sort_values('duration_int', ascending=False)[['title', 'duration_int']].head(10)
plt.figure()
sns.barplot(x='duration_int', y='title', data=longest, palette='flare')
plt.title('Top 10 Longest Movies')
plt.xlabel('Duration (min)')
plt.savefig('images/13_longest_movies.png', bbox_inches='tight')
plt.show()


In [ ]:
# 14. TV Show seasons distribution
tv = df[df['type'] == 'TV Show']
plt.figure()
sns.countplot(x=tv['duration_int'], palette='Set3')
plt.title('Number of Seasons - TV Shows')
plt.xlabel('Seasons')
plt.savefig('images/14_tvshow_seasons.png', bbox_inches='tight')
plt.show()


In [ ]:
# 15. Month added distribution (Count Plot)
plt.figure()
order = ['January','February','March','April','May','June','July',
         'August','September','October','November','December']
sns.countplot(y=df['month_added'], order=order, palette='cubehelix')
plt.title('Content Added by Month')
plt.savefig('images/15_month_added.png', bbox_inches='tight')
plt.show()


## 8. Key Insights

> Replace each bullet below with your own finding once you've run the cells above.
> Aim for 15-20 insights total, phrased as data-backed statements, e.g.:
> "More than 65% of Netflix content is rated TV-MA or TV-14, showing the platform
> primarily targets mature audiences."

- Insight 1: ...
- Insight 2: ...
- Insight 3: ...
- Insight 4: ...
- Insight 5: ...
- Insight 6: ...
- Insight 7: ...
- Insight 8: ...
- Insight 9: ...
- Insight 10: ...


## 9. Conclusion

Summarize in 3-5 sentences: what the data shows overall, which findings were most
surprising, and what you'd explore next (e.g. sentiment analysis on descriptions,
predicting content popularity, etc.).
